# RAG and Context Memory

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

from langchain.prompts import PromptTemplate, ChatPromptTemplate, MessagesPlaceholder

from langchain_community.document_loaders import TextLoader
from langchain.vectorstores import Chroma

from langchain.text_splitter import CharacterTextSplitter # split document based on characters.
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain.chains import RetrievalQA, ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory
# `ConversationChain` and `ConversationBufferMemory` is old fashioned now.
# USE RunnableWithMessageHistory, which is new Langchain standard

from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables import RunnableWithMessageHistory

from IPython.display import display, HTML, Markdown

import wget

In [2]:
# Model configuration
openai_llm = ChatOpenAI(
    model="gpt-4.1-nano",
)

## Preprocessing

### Load the document

In [3]:
filename = 'companyPolicies.txt'

In [ ]:
# Download and read content of a text file

# url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/6JDbUb_L3egv_eOkouY71A.txt'

# # Use wget to download the file
# wget.download(url, out=filename)
# print('file downloaded')

# with open(filename, 'r') as file:
#     # Read the contents of the file
#     contents = file.read()
#     print(contents)

### Splitting the document into chunks

In [4]:
loader = TextLoader(filename)
documents = loader.load() # It converted our text file into one document
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts = text_splitter.split_documents(documents)
print(len(texts))

Created a chunk of size 1624, which is longer than the specified 1000
Created a chunk of size 1885, which is longer than the specified 1000
Created a chunk of size 1903, which is longer than the specified 1000
Created a chunk of size 1729, which is longer than the specified 1000
Created a chunk of size 1678, which is longer than the specified 1000
Created a chunk of size 2032, which is longer than the specified 1000
Created a chunk of size 1894, which is longer than the specified 1000


16


In [5]:
documents

[Document(metadata={'source': 'companyPolicies.txt'}, page_content="1.\tCode of Conduct\n\nOur Code of Conduct outlines the fundamental principles and ethical standards that guide every member of our organization. We are committed to maintaining a workplace that is built on integrity, respect, and accountability.\nIntegrity: We hold ourselves to the highest ethical standards. This means acting honestly and transparently in all our interactions, whether with colleagues, clients, or the broader community. We respect and protect sensitive information, and we avoid conflicts of interest.\nRespect: We embrace diversity and value each individual's contributions. Discrimination, harassment, or any form of disrespectful behavior is unacceptable. We create an inclusive environment where differences are celebrated and everyone is treated with dignity and courtesy.\nAccountability: We take responsibility for our actions and decisions. We follow all relevant laws and regulations, and we strive to 

### Embedding and storing

In [5]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
docsearch = Chroma.from_documents(texts, embeddings)  # store the embedding in docsearch using Chromadb
print('document ingested')

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


document ingested


## RetrievalQA

In [ ]:
qa = RetrievalQA.from_chain_type(llm=openai_llm, 
                                 chain_type="stuff", # "stuff" means all retrieved documents are simply concatenated and passed to the LLM
                                 retriever=docsearch.as_retriever(), 
                                 return_source_documents=True, # return source documents too
                                 )
query = "what is mobile policy?"
qa.invoke(query)

{'query': 'what is mobile policy?',
 'result': 'The mobile policy outlines the standards and expectations for the responsible and appropriate use of mobile devices within the organization. It covers aspects such as acceptable use, security measures, confidentiality, cost management, compliance with laws, procedures for reporting lost or stolen devices, and the consequences of non-compliance. The goal is to ensure that employees use mobile phones in a manner that aligns with company values, legal requirements, and security best practices.',
 'source_documents': [Document(metadata={'source': 'companyPolicies.txt'}, page_content='4.\tMobile Phone Policy'),
  Document(metadata={'source': 'companyPolicies.txt'}, page_content='The Mobile Phone Policy sets forth the standards and expectations governing the appropriate and responsible usage of mobile devices in the organization. The purpose of this policy is to ensure that employees utilize mobile phones in a manner consistent with company val

In [16]:
qa = RetrievalQA.from_chain_type(llm=openai_llm, 
                                 chain_type="stuff", 
                                 retriever=docsearch.as_retriever(), 
                                 return_source_documents=False)
query = "Can I eat in company vehicles?"
qa.invoke(query)

{'query': 'Can I eat in company vehicles?',
 'result': 'The provided policies do not explicitly mention whether eating in company vehicles is permitted or prohibited.'}

## Provide System message

Using prompt template

In [29]:
prompt_template = """Use the information from the document to answer the question at the end. If you don't know the answer, just say that you don't know, definately do not try to make up an answer.

{context}

Question: {question}
"""

PROMPT = PromptTemplate(
    template=prompt_template, input_variables=["context", "question"]
)

chain_type_kwargs = {"prompt": PROMPT}

In [30]:
qa = RetrievalQA.from_chain_type(llm=openai_llm, 
                                 chain_type="stuff", 
                                 retriever=docsearch.as_retriever(), 
                                 chain_type_kwargs=chain_type_kwargs, 
                                 return_source_documents=False)

query = "Can I eat in company vehicles?"
qa.invoke(query)

{'query': 'Can I eat in company vehicles?',
 'result': 'Based on the provided policies, smoking is not permitted in company vehicles, but there is no specific mention of restrictions on eating in company vehicles. Therefore, I do not know whether eating in company vehicles is allowed or not.'}

## Make the conversation have memory

In [ ]:
query = "What I cannot do in it?" # Here we continued previous conversation and "it" means "vehicle". LLM don't know context memory yet.
                                    # So it returned with unexpected answer.
qa.invoke(query)

{'query': 'What I cannot do in it?',
 'result': "Based on the provided policies, you cannot engage in the following activities:\n\n- Use company internet and email for non-job-related purposes during work hours, especially if it interferes with responsibilities.\n- Share your login credentials or passwords.\n- Open email attachments or click links from unknown or untrusted sources without caution.\n- Transmit confidential or sensitive information via unsecured channels.\n- Engage in harassment, discrimination, or distribute offensive or inappropriate content online.\n- Use internet or email to violate laws, regulations, or copyright.\n- Use mobile devices in a way that disrupts work or violates security protocols.\n- Share sensitive company information through unsecured messaging apps or emails.\n- Disclose company information publicly on social media or forums without discretion.\n- Use company mobile phones for personal charges without reimbursement.\n- Engage in discrimination or ha

In [6]:
memory = ConversationBufferMemory(memory_key = "chat_history", return_message = True) # To make the LLM have memory

In [7]:
def get_chat_history(h):
    print("------------ History -------------")
    print(h)
    return h

In [8]:
# Create a ConversationalRetrievalChain to retrieve information and talk with the LLM.
qa = ConversationalRetrievalChain.from_llm(llm=openai_llm, 
                                           chain_type="stuff", 
                                           retriever=docsearch.as_retriever(), 
                                           memory = memory,
                                        #    get_chat_history=lambda h : h,
                                           get_chat_history=get_chat_history,
                                           return_source_documents=False)

In [9]:
history = []

In [ ]:
query = "What is mobile policy?"
result = qa.invoke({"question":query})
print("---------- RESULT -------------")
print("QUERY: ", result["question"])
print("ANSWER: ", result["answer"])

------------ History -------------



Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


---------- RESULT -------------
QUERY:  What is mobile policy?
ANSWER:  The Mobile Phone Policy sets standards and expectations for the appropriate and responsible use of mobile devices within the organization. It emphasizes that mobile devices are mainly intended for work-related tasks, with limited personal use allowed as long as it does not disrupt work. The policy also highlights the importance of safeguarding devices and access credentials, avoiding transmitting sensitive information via unsecured channels, keeping personal charges separate from company accounts, and complying with relevant laws and regulations. Additionally, it requires immediate reporting of lost or stolen devices and outlines potential disciplinary actions for non-compliance. The overall goal is to promote responsible, secure, and ethical mobile device usage aligned with company values and legal standards.


In [ ]:
query = "List points in it?" # It represents "Mobile policy" here
result = qa.invoke({"question":query})
print("---------- RESULT -------------")
print("QUERY: ", result["question"])
print("ANSWER: ", result["answer"])

------------ History -------------
Human: What is mobile policy?
AI: The Mobile Phone Policy sets standards and expectations for the appropriate and responsible use of mobile devices within the organization. It emphasizes that mobile devices are mainly intended for work-related tasks, with limited personal use allowed as long as it does not disrupt work. The policy also highlights the importance of safeguarding devices and access credentials, avoiding transmitting sensitive information via unsecured channels, keeping personal charges separate from company accounts, and complying with relevant laws and regulations. Additionally, it requires immediate reporting of lost or stolen devices and outlines potential disciplinary actions for non-compliance. The overall goal is to promote responsible, secure, and ethical mobile device usage aligned with company values and legal standards.
---------- RESULT -------------
QUERY:  List points in it?
ANSWER:  The key points outlined in the Mobile Pho

`get_chat_history` internally manages history. And llm now knows the context and determine `it` means `Mobile policy`.

## Manually pass history to LLM

In [ ]:
history = []
qa = ConversationalRetrievalChain.from_llm(llm=openai_llm, 
                                           chain_type="stuff", 
                                           retriever=docsearch.as_retriever(), 
                                           memory=None,
                                        #    get_chat_history=lambda h : h,
                                           get_chat_history=get_chat_history,
                                           return_source_documents=False)

In [17]:
# Picked query and answer from one of the previous llm calls
query="Can I eat in company vehicles?"
answer="Based on the provided policies, smoking is not permitted in company vehicles, but there is no specific mention of restrictions on eating in company vehicles. Therefore, I do not know whether eating in company vehicles is allowed or not."

history.append((query, answer)) # Append the query and answer to the history.

In [18]:
history

[('Can I eat in company vehicles?',
  'Based on the provided policies, smoking is not permitted in company vehicles, but there is no specific mention of restrictions on eating in company vehicles. Therefore, I do not know whether eating in company vehicles is allowed or not.')]

In [20]:
query = "What I cannot do in it?" # it represents "vehicles" here
result = qa.invoke({"question":query, "chat_history": history}) # manually passing history to llm call
print("---------- RESULT -------------")
print("QUERY: ", result["question"])
print("ANSWER: ", result["answer"])

------------ History -------------
[('Can I eat in company vehicles?', 'Based on the provided policies, smoking is not permitted in company vehicles, but there is no specific mention of restrictions on eating in company vehicles. Therefore, I do not know whether eating in company vehicles is allowed or not.')]
---------- RESULT -------------
QUERY:  What I cannot do in it?
ANSWER:  You are not allowed to smoke in company vehicles, whether they are owned or leased.


## RunnableWithMessageHistory for managing history

Instead of managing history in `ConversationalRetrievalChain`, we will manage it via `RunnableWithMessageHistory`

In [ ]:
# Session-based message history store

store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory() # Way to pass/prevent history
    return store[session_id]

qa_chain = ConversationalRetrievalChain.from_llm(
    llm=openai_llm,
    retriever=docsearch.as_retriever(),
    chain_type="stuff",
    return_source_documents=False
)

qa_with_history = RunnableWithMessageHistory(
    qa_chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="chat_history",
)


In [31]:
result = qa_with_history.invoke(
    {"question": "What is mobile policy?"},
    config={"configurable": {"session_id": "session_0"}}
)
print("----------- RESULT-1 --------------")
print(result)

result = qa_with_history.invoke(
    {"question": "List points in it?"},
    config={"configurable": {"session_id": "session_0"}}
)
print("----------- RESULT-2 --------------")
print(result['question'])
print(result['chat_history'])
print(result['answer'])


----------- RESULT-1 --------------
{'question': 'What is mobile policy?', 'chat_history': [], 'answer': 'The Mobile Phone Policy sets standards and expectations for the appropriate and responsible use of mobile devices within the organization. It emphasizes that mobile devices should primarily be used for work-related tasks, with limited personal use that does not interfere with work obligations. The policy also highlights the importance of security measures, such as safeguarding devices and access credentials, being cautious when downloading apps or clicking links from unknown sources, and reporting security concerns promptly. It advises employees to avoid transmitting sensitive company information through unsecured messaging apps or emails and to be discreet when discussing company matters publicly. Additionally, the policy covers cost management by requiring employees to keep personal phone usage separate from company accounts and reimburse any personal charges on company-issued ph

## Wrap up and make it an agent


In [34]:
while True:
    query = input("Question: ")
    
    if query.lower() in ["quit","exit","bye"]:
        print("Answer: Goodbye!")
        break
        
    result = qa_with_history.invoke(
                                    {"question": query}, 
                                    config={"configurable": {"session_id": "chat_0"}}
                                )
    print("Question: ", result["question"])
    print("Answer: ", result["answer"])

Question:  Whats policy about vehicle?
Answer:  The policy regarding vehicles states that smoking is not permitted in company vehicles, whether they are owned or leased, in order to maintain the condition and cleanliness of these vehicles.
Question:  list prohibited things in it
Answer:  The vehicle policy specifically prohibits smoking in company vehicles, whether they are owned or leased.
Answer: Goodbye!
